In [6]:
from pyspark.sql import SparkSession

In [25]:
spark = SparkSession.builder \
        .appName("RDD") \
        .master("local[8]") \
        .getOrCreate()

In [26]:
spark

In [24]:
spark.stop()

In [27]:
sentenses = ["first element in the list", "hello from spark", "create spark rdd", "last element in the list"]

In [28]:
sc = spark.sparkContext

In [48]:
raw_rdd = sc.parallelize(sentenses*1000000)

In [30]:
raw_rdd.collect()

['first element in the list',
 'hello from spark',
 'create spark rdd',
 'last element in the list']

In [31]:
print(raw_rdd.id())

0


In [49]:
transformed_rdd = raw_rdd.map(lambda x: x.split(" "))

In [51]:
transformed_rdd.take(10)

['first', 'element', 'in', 'the', 'list']

In [34]:
print(transformed_rdd.id())

1


In [52]:
transformed_rdd = raw_rdd.flatMap(lambda x: x.split(" "))

In [37]:
transformed_rdd.collect()

['first',
 'element',
 'in',
 'the',
 'list',
 'hello',
 'from',
 'spark',
 'create',
 'spark',
 'rdd',
 'last',
 'element',
 'in',
 'the',
 'list']

In [38]:
print(transformed_rdd.id())

2


In [39]:
print(transformed_rdd.toDebugString().decode())

(8) PythonRDD[2] at collect at /tmp/ipykernel_421/2149301238.py:1 []
 |  ParallelCollectionRDD[0] at readRDDFromFile at PythonRDD.scala:289 []


In [53]:
words_rdd = transformed_rdd.map(lambda x: (x, 1))

In [41]:
words_rdd.collect()

[('first', 1),
 ('element', 1),
 ('in', 1),
 ('the', 1),
 ('list', 1),
 ('hello', 1),
 ('from', 1),
 ('spark', 1),
 ('create', 1),
 ('spark', 1),
 ('rdd', 1),
 ('last', 1),
 ('element', 1),
 ('in', 1),
 ('the', 1),
 ('list', 1)]

In [42]:
print(words_rdd.toDebugString().decode())

(8) PythonRDD[3] at RDD at PythonRDD.scala:53 []
 |  ParallelCollectionRDD[0] at readRDDFromFile at PythonRDD.scala:289 []


In [54]:
%%time
grouped_rdd_byKey = words_rdd.groupByKey().mapValues(lambda values: sum(values))
grouped_rdd_byKey.collect()

CPU times: user 43 ms, sys: 9.86 ms, total: 52.8 ms
Wall time: 3.44 s


[('last', 1000000),
 ('in', 2000000),
 ('the', 2000000),
 ('first', 1000000),
 ('spark', 2000000),
 ('element', 2000000),
 ('list', 2000000),
 ('create', 1000000),
 ('hello', 1000000),
 ('from', 1000000),
 ('rdd', 1000000)]

In [55]:
%%time

grouped_rdd_reduceByKey = words_rdd.reduceByKey(lambda x,y: x+y)
grouped_rdd_reduceByKey.collect()

CPU times: user 20.8 ms, sys: 14.9 ms, total: 35.7 ms
Wall time: 2.26 s


[('last', 1000000),
 ('in', 2000000),
 ('the', 2000000),
 ('first', 1000000),
 ('spark', 2000000),
 ('element', 2000000),
 ('list', 2000000),
 ('create', 1000000),
 ('hello', 1000000),
 ('from', 1000000),
 ('rdd', 1000000)]

In [45]:
print(grouped_rdd_reduceByKey.toDebugString().decode())

(8) PythonRDD[13] at collect at <timed exec>:2 []
 |  MapPartitionsRDD[12] at mapPartitions at PythonRDD.scala:160 []
 |  ShuffledRDD[11] at partitionBy at NativeMethodAccessorImpl.java:0 []
 +-(8) PairwiseRDD[10] at reduceByKey at <timed exec>:1 []
    |  PythonRDD[9] at reduceByKey at <timed exec>:1 []
    |  ParallelCollectionRDD[0] at readRDDFromFile at PythonRDD.scala:289 []


In [46]:
words_rdd.cache()
words_rdd.collect()

[('first', 1),
 ('element', 1),
 ('in', 1),
 ('the', 1),
 ('list', 1),
 ('hello', 1),
 ('from', 1),
 ('spark', 1),
 ('create', 1),
 ('spark', 1),
 ('rdd', 1),
 ('last', 1),
 ('element', 1),
 ('in', 1),
 ('the', 1),
 ('list', 1)]

In [58]:
grouped_rdd_reduceByKey.first()

In [57]:
print(grouped_rdd_reduceByKey.getNumPartitions())

8


In [60]:
employees_rdd = sc.textFile("file:///home/ITI/rdd/emp_clean.txt")

In [61]:
print(employees_rdd.getNumPartitions())

2


In [62]:
employees_rdd.collect()

['emp_id,name,department,job_title,salary,location,hire_date,performance_rating,years_exp',
 '1,John Smith,Engineering,Senior Developer,125000,San Francisco,2021-03-15,4.5,8',
 '2,Sarah Johnson,Sales,Account Executive,85000,New York,2022-01-10,4.2,5',
 '3,Michael Williams,Engineering,Software Engineer,95000,Austin,2023-06-20,3.8,3',
 '4,Jennifer Brown,Marketing,Marketing Manager,92000,Chicago,2020-11-05,4.7,7',
 '5,David Jones,Finance,Senior Analyst,105000,Boston,2021-08-12,4.3,6',
 '6,Lisa Garcia,IT,DevOps Engineer,115000,Seattle,2022-04-18,4.6,4',
 '7,Robert Martinez,Legal,Legal Counsel,145000,San Francisco,2019-09-22,4.8,10',
 '8,Patricia Wilson,HR,HR Manager,88000,Denver,2020-02-28,4.1,6',
 '9,James Anderson,Sales,Sales Manager,110000,Atlanta,2021-12-01,4.4,7',
 '10,Mary Thomas,Engineering,Tech Lead,145000,Seattle,2018-07-14,4.9,12']

In [63]:
header = employees_rdd.first()

In [64]:
emp_rdd = employees_rdd.filter(lambda x: x != header)

In [66]:
emp_rdd.take(3)

['1,John Smith,Engineering,Senior Developer,125000,San Francisco,2021-03-15,4.5,8',
 '2,Sarah Johnson,Sales,Account Executive,85000,New York,2022-01-10,4.2,5',
 '3,Michael Williams,Engineering,Software Engineer,95000,Austin,2023-06-20,3.8,3']

In [69]:
lines_rdd = emp_rdd.map(lambda x : x.split(','))
lines_rdd.take(2)

[['1',
  'John Smith',
  'Engineering',
  'Senior Developer',
  '125000',
  'San Francisco',
  '2021-03-15',
  '4.5',
  '8'],
 ['2',
  'Sarah Johnson',
  'Sales',
  'Account Executive',
  '85000',
  'New York',
  '2022-01-10',
  '4.2',
  '5']]

In [72]:
departments_rdd = lines_rdd.map(lambda x: (x[2], 1))

In [73]:
departments_rdd.collect()

[('Engineering', 1),
 ('Sales', 1),
 ('Engineering', 1),
 ('Marketing', 1),
 ('Finance', 1),
 ('IT', 1),
 ('Legal', 1),
 ('HR', 1),
 ('Sales', 1),
 ('Engineering', 1)]

In [74]:
departments_count = departments_rdd.reduceByKey(lambda x,y : x+y)

In [75]:
departments_count.collect()

[('Engineering', 3),
 ('Sales', 2),
 ('Finance', 1),
 ('IT', 1),
 ('Legal', 1),
 ('HR', 1),
 ('Marketing', 1)]

In [79]:
emp_rdd_1 = lines_rdd.map(lambda x: (x[0], (x[1],x[3])))

In [80]:
emp_rdd_1.take(3)

[('1', ('John Smith', 'Senior Developer')),
 ('2', ('Sarah Johnson', 'Account Executive')),
 ('3', ('Michael Williams', 'Software Engineer'))]

In [81]:
emp_rdd_2 = lines_rdd.map(lambda x: (x[0], x[4]))

In [82]:
emp_rdd_2.take(3)

[('1', '125000'), ('2', '85000'), ('3', '95000')]

In [83]:
emp_rdd_joined = emp_rdd_1.join(emp_rdd_2)

In [84]:
emp_rdd_joined.collect()

[('4', (('Jennifer Brown', 'Marketing Manager'), '92000')),
 ('5', (('David Jones', 'Senior Analyst'), '105000')),
 ('10', (('Mary Thomas', 'Tech Lead'), '145000')),
 ('1', (('John Smith', 'Senior Developer'), '125000')),
 ('2', (('Sarah Johnson', 'Account Executive'), '85000')),
 ('9', (('James Anderson', 'Sales Manager'), '110000')),
 ('3', (('Michael Williams', 'Software Engineer'), '95000')),
 ('6', (('Lisa Garcia', 'DevOps Engineer'), '115000')),
 ('7', (('Robert Martinez', 'Legal Counsel'), '145000')),
 ('8', (('Patricia Wilson', 'HR Manager'), '88000'))]

In [86]:
emp_rdd_joined.map(lambda x: (x[1][0][0], x[1][0][1], x[1][1])).collect()

[('Jennifer Brown', 'Marketing Manager', '92000'),
 ('David Jones', 'Senior Analyst', '105000'),
 ('Mary Thomas', 'Tech Lead', '145000'),
 ('John Smith', 'Senior Developer', '125000'),
 ('Sarah Johnson', 'Account Executive', '85000'),
 ('James Anderson', 'Sales Manager', '110000'),
 ('Michael Williams', 'Software Engineer', '95000'),
 ('Lisa Garcia', 'DevOps Engineer', '115000'),
 ('Robert Martinez', 'Legal Counsel', '145000'),
 ('Patricia Wilson', 'HR Manager', '88000')]